In [0]:
%sql
use catalog capstone4_retail;
use schema retail_orders;

In [0]:
ROOT="/Volumes/capstone4_retail/retail_orders/ext_capstone4_vol"
display(dbutils.fs.ls(ROOT))
display(dbutils.fs.ls(ROOT + "/status_updates"))
display(dbutils.fs.ls(ROOT+"/primary_events"))
display(dbutils.fs.ls(ROOT + "/secondary_events"))


In [0]:
directories = [
    "checkpoints/primary",
    "checkpoints/status",
    "checkpoints/secondary",
    "schema_tracking/primary",
    "schema_tracking/status",
    "schema_tracking/secondary",
    "logs",
    "quarantine"
]
for directory in directories:
    dbutils.fs.mkdirs(ROOT + "/" + directory)

print("All project directories created successfully")

In [0]:
from pyspark.sql.functions import lit

PRIMARY_PATH = ROOT + "/primary_events"

files = dbutils.fs.ls(PRIMARY_PATH)

primary_dfs = []

for file in files:

    if file.name.endswith(".csv"):

        temp_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(file.path)
        )

        if "source_schema_version" not in temp_df.columns:

            temp_df = temp_df.withColumn(
                "source_schema_version",
                lit(None).cast("string")
            )

        primary_dfs.append(temp_df)

primary_df = primary_dfs[0]

for temp_df in primary_dfs[1:]:

    primary_df = primary_df.unionByName(
        temp_df,
        allowMissingColumns=True
    )

print("Primary records:", primary_df.count())

display(primary_df.limit(10))

In [0]:
primary_df.printSchema()

In [0]:
total_records = primary_df.count()

unique_sale_ids = (
    primary_df
    .select("sale_id")
    .distinct()
    .count()
)

print("Total records:", total_records)
print("Unique sale IDs:", unique_sale_ids)
print("Repeated sale IDs:", total_records - unique_sale_ids)

In [0]:
from pyspark.sql.functions import count

duplicate_df = (
    primary_df
    .groupBy("sale_id")
    .agg(
        count("*").alias("record_count")
    )
    .filter("record_count > 1")
)

display(duplicate_df.orderBy("sale_id"))

In [0]:
display(
    primary_df
    .groupBy("order_status")
    .count()
    .orderBy("order_status")
)

In [0]:
from pyspark.sql.functions import (
    min,
    max,
    avg,
    percentile_approx,col
)

metric_df = primary_df.withColumn(
    "sales_amount_num",
    col("sales_amount").cast("decimal(18,2)")
)

display(
    metric_df.select(
        min("sales_amount_num").alias("minimum_sales"),
        max("sales_amount_num").alias("maximum_sales"),
        avg("sales_amount_num").alias("average_sales"),
        percentile_approx(
            "sales_amount_num", 0.50
        ).alias("median_sales"),
        percentile_approx(
            "sales_amount_num", 0.95
        ).alias("p95_sales")
    )
)
display(
    metric_df
    .orderBy(col("sales_amount_num").desc())
    .select(
        "sale_id",
        "sales_amount",
        "quantity",
        "store_id",
        "product_id"
    )
    .limit(50)
)

In [0]:
from pyspark.sql.functions import regexp_extract

batch_results = []

for file in files:

    if file.name.endswith(".csv"):

        batch_df = (
            spark.read
            .option("header", True)
            .csv(file.path)
        )

        batch_number = file.name.replace(
            "sale_batch_", ""
        ).replace(".csv", "")

        batch_results.append(
            (batch_number, batch_df.count())
        )

batch_report = spark.createDataFrame(
    batch_results,
    ["batch_number", "record_count"]
)

display(batch_report.orderBy("batch_number"))

In [0]:
STATUS_PATH = ROOT + "/status_updates"
SECONDARY_PATH = ROOT + "/secondary_events"

status_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(STATUS_PATH)
)

secondary_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(SECONDARY_PATH)
)

print("Status records:", status_df.count())
print("Secondary event records:", secondary_df.count())

display(status_df.limit(10))
display(secondary_df.limit(10))
display(
    status_df
    .groupBy("previous_status", "new_status")
    .count()
    .orderBy("previous_status", "new_status")
)
display(
    secondary_df
    .groupBy("event_type")
    .count()
    .orderBy("event_type")
)

In [0]:
REFERENCE_PATH = ROOT + "/reference"

store_csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(REFERENCE_PATH + "/store_id.csv")
)

product_csv_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(REFERENCE_PATH + "/product_id.csv")
)

print("Store records:", store_csv_df.count())
print("Product records:", product_csv_df.count())

display(store_csv_df.limit(5))
display(product_csv_df.limit(5))